# SeLMRoute paper reproduction

This notebook reproduces the statistical experiments from the frozen public data. It does **not** call JEV, Laya, Hugging Face, or any other external model/API. The CLI equivalent is `uv run selmroute reproduce`.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
ROOT

PosixPath('/home/federated/PycharmProjects/selmroute')

## 1. Validate the frozen release

The release must contain exactly 11,481 performance samples, 229,620 performance outcomes, 12,446 cost samples, 161,520 cost outcomes, the frozen semantic features/embeddings, JEV-Direct predictions, and both released CatBoost bundles.

In [2]:
from selmroute.validation import validate_release

validation = validate_release(ROOT)
validation['performance'], validation['cost']

({'samples': 11481,
  'datasets': 15,
  'models': 20,
  'outcomes': 229620,
  'jev': {'rows': 11481, 'probability_mass_features': 40},
  'laya': {'rows': 11481, 'probability_mass_features': 40},
  'gte_dimension': 3584,
  'gte_model': {'model_name': 'Alibaba-NLP/gte-Qwen2-7B-instruct',
   'revision': 'a8d08b36ada9cacfe34c4d6f80957772a025daf2'}},
 {'samples': 12446,
  'datasets': 10,
  'models': 13,
  'outcomes': 161520,
  'baseline': 'gpt-5'})

## 2. Re-run the experiments

All semantic evidence is loaded from the published files. No semantic backend inference is performed.

In [ ]:
from selmroute.reproduce import reproduce_all

summary = reproduce_all(ROOT, ROOT / 'artifacts/reproduction')
summary['note']

## 3. Main grouped results

In [3]:
main = pd.read_csv(ROOT / 'artifacts/reproduction/performance_grouped/paper/main_table/table1_complete.csv')
main

,system,n_seeds,AvgAcc_mean,AvgAcc_std,Gain@R_mean,Gain@R_std,Gain@B_mean,Gain@B_std,Gap@O_mean,Gap@O_std,BestSingleAvg_mean,BestSingleAvg_std,DatasetOracleAvg_mean,DatasetOracleAvg_std,OracleAvg_mean,OracleAvg_std,best_single_model
0,probability_mass,5,72.078064,0.448553,56.081978,1.616994,5.721481,0.666788,22.001600,0.667504,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B
1,full,5,71.300339,0.178203,54.108095,1.395193,4.575067,0.519520,22.840500,0.439849,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B
2,hard,5,71.200595,0.585196,54.607545,2.283153,4.410173,1.111454,22.943249,0.865717,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B
3,gte_catboost,5,71.161611,0.738386,53.316353,2.711476,4.341070,0.929817,23.013846,0.902761,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B
4,domain_only,5,71.081746,1.031335,53.447692,2.186634,4.454549,0.821745,23.158802,1.132890,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B
5,tfidf,5,70.869698,0.775038,53.254728,1.175557,3.819727,1.048649,23.287820,1.246902,68.808062,0.343677,73.938802,0.506225,91.911864,0.42567,Qwen3-8B


## 4. Grouped OOF and paired tests

In [4]:
perf = ROOT / 'artifacts/reproduction/performance_grouped/paper'
pm = json.loads((perf / 'cv_grouped/probability_mass/oof_metrics.json').read_text())
hard = json.loads((perf / 'cv_grouped/hard/oof_metrics.json').read_text())
laya = json.loads((perf / 'laya_grouped/cv/oof_metrics.json').read_text())
pm_hard = json.loads((perf / 'stats/probability_mass_vs_hard.json').read_text())
{'ProbabilityMass': pm['AvgAcc'], 'Hard': hard['AvgAcc'], 'Laya': laya['AvgAcc'], 'PM-Hard pp': pm_hard['macro_diff_pp'], 'p': pm_hard['permutation_p_two_sided']}

{'ProbabilityMass': 72.63595789619457,
 'Hard': 71.80974258384111,
 'Laya': 70.53394288166791,
 'PM-Hard pp': 0.8262153123534615,
 'p': 0.10989450527473626}

## 5. Nested Lite, OOD, probe ablation, and cost

In [5]:
nested = json.loads(
    (perf / 'nested_lite12_grouped/nested_lite_metrics.json').read_text()
)

ood = pd.read_csv(
    ROOT / 'artifacts/reproduction/ood/summary.csv'
)

probe = pd.read_csv(
    perf / 'leave_one_probe_out/leave_one_probe_out_summary.csv'
)

cost = pd.read_csv(
    ROOT / 'artifacts/reproduction/performance_cost/summary.csv'
)

print('Nested Lite:', nested)
display(ood)
display(probe[['system', 'AvgAcc_mean', 'drop_vs_all_pp']])
display(
    cost[
        [
            'seed',
            'baseline_AvgAcc',
            'theta_star_AvgAcc',
            'PerfGain_pct',
            'CostSave_pct',
        ]
    ]
)

Nested Lite: {'method': 'nested_leave_one_probe_out_top_k', 'router': 'catboost', 'representation': 'probability_mass', 'outer_folds': 5, 'inner_folds': 3, 'seed': 3407, 'keep_probes': 12, 'available_probes': 16, 'full_feature_count': 40, 'lite_feature_count_mean': 28.2, 'full16_AvgAcc': 72.63595789619457, 'nested_lite_AvgAcc': 72.49140782681374, 'nested_lite_minus_full_pp': -0.14455006938082704, 'paired_bootstrap_ci_low_pp': -0.9032812938518826, 'paired_bootstrap_ci_high_pp': 0.6141450041278932, 'paired_permutation_p_two_sided': 0.7058147092645368, 'noninferiority_margin_pp': 0.5, 'noninferiority_lower_95_one_sided_pp': -0.7833005239759353, 'noninferior_to_full16': False, 'n_queries': 11481, 'n_datasets': 15, 'group_duplicates': True}


,protocol,system,seed,AvgAcc,Gain@R,Gain@B,Gap@O,BestSingleAvg,DatasetOracleAvg,OracleAvg,best_single_model,n_datasets,n_test
0,dataset-ood,probability_mass,3407,67.021460,44.078171,-3.261544,27.584461,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
1,dataset-ood,full,3407,67.273654,44.373655,-2.618782,27.347164,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
2,dataset-ood,gte_catboost,3407,67.290190,44.656384,-2.911369,27.257322,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
3,dataset-ood,tfidf,3407,66.676707,43.320180,-3.534585,27.971820,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
4,dataset-ood,domain_only,3407,65.935017,39.719620,-4.234223,28.985349,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
5,domain-ood,probability_mass,3407,65.658442,42.668673,-6.101916,28.964411,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
6,domain-ood,full,3407,65.436343,42.432513,-6.148738,29.170799,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
7,domain-ood,gte_catboost,3407,67.306544,46.195184,-3.254943,27.172266,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
8,domain-ood,tfidf,3407,65.618022,41.748345,-5.514323,28.984764,69.226674,73.150094,91.855093,Qwen3-8B,15,11481
9,domain-ood,domain_only,3407,65.636771,42.125174,-5.842239,29.072623,69.226674,73.150094,91.855093,Qwen3-8B,15,11481


,system,AvgAcc_mean,drop_vs_all_pp
0,drop_domain_specialization,72.267589,-0.189525
1,drop_ambiguity,72.262052,-0.183988
2,drop_math_reasoning,72.257146,-0.179081
3,drop_code_reasoning,72.252473,-0.174409
4,drop_social_affective,72.149871,-0.071807
5,drop_decomposition_need,72.106321,-0.028257
6,all_probes,72.078064,0.000000
7,drop_external_knowledge,72.014215,0.063850
8,drop_reasoning_depth,71.960054,0.118010
9,drop_exactness,71.950530,0.127534


,seed,baseline_AvgAcc,theta_star_AvgAcc,PerfGain_pct,CostSave_pct
0,42,65.634782,67.846385,3.369559,NaN
1,3407,65.911438,68.930208,4.580038,-0.095079
2,0,64.379042,66.916392,3.941267,NaN
3,1,64.917774,65.480302,0.866524,NaN
4,2,66.777788,67.124951,0.519878,-1.582653
